# Hierarchical Clustering — Agglomerative Clustering (Day 1 Scaffold)
## 23CSE301 ML Capstone · Review 2 · Clustering Track
---
**Problem Context:**  
Apply Agglomerative Hierarchical Clustering to the AI4I 2020 Predictive Maintenance dataset  
to discover natural machine operating condition groupings, **without** using ground-truth failure labels.  
Ground-truth labels (`Machine failure`) are retained separately for post-hoc evaluation only.

**Day 1 Scope:**
- Feature preparation and scaling (consistent with K-Means notebook for fair comparison)
- Agglomerative Clustering implementation (Ward linkage, full 10,000-row dataset)
- Practical dendrogram workflow (500-sample documented subset)
- Linkage comparison: Ward vs Average vs Complete (Silhouette, Davies-Bouldin, Calinski-Harabasz)
- Cluster label generation
- Day 1 validation checks

**Day 2 scope (pending):** Metrics across k values, PCA plot, t-SNE, K-Means comparison table.  
**Day 3 scope (pending):** Final notebook, additional visualisations, consolidated analysis.

> [STUDENT TASK] prompts require your **original analysis**. Do not paraphrase AI-written answers.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os

warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports complete.")

## 2. Dataset Loading and Feature Preparation

The dataset is loaded from the canonical Review 2 data directory.  
**Key data-leakage prevention:** `Machine failure` and all failure sub-type columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) are excluded from clustering features.  
These columns are kept separately as `TRUE_LABELS` for post-hoc evaluation only — they are **never** passed to the clustering model.


In [ ]:
# Notebook is at: Review2/notebooks/clustering/
# Dataset is at:  Review2/data/ai4i2020.csv
# Relative path from this notebook: ../../data/ai4i2020.csv
DATA_PATH = os.path.join('..', '..', 'data', 'ai4i2020.csv')

# Fallback: if running from the Review2/ root directly
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join('data', 'ai4i2020.csv')

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found. Expected at: {os.path.abspath(DATA_PATH)}\n"
        "Run this notebook from Review2/notebooks/clustering/ or Review2/ directory."
    )

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
df['Type_encoded'] = df['Type'].map({'L': 0, 'M': 1, 'H': 2})

# Same feature set as K-Means for fair algorithm comparison
CLUSTER_FEATURES = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Type_encoded',
]

# Ground-truth labels — for POST-HOC evaluation only; NEVER used in model fitting
TRUE_LABELS = df['Machine failure'].values

X_cluster = df[CLUSTER_FEATURES].copy()
cluster_scaler = StandardScaler()
X_scaled = cluster_scaler.fit_transform(X_cluster)

print(f"Dataset: {df_raw.shape}")
print(f"Clustering feature matrix: {X_scaled.shape}")
print(f"Features: {CLUSTER_FEATURES}")
print()
print("KEY CHECK: 'Machine failure' is NOT in CLUSTER_FEATURES:", 'Machine failure' not in CLUSTER_FEATURES)
print(f"Failure rate in dataset: {TRUE_LABELS.mean()*100:.2f}%  (for post-hoc evaluation only)")

## 3. Dendrogram — Large Dataset Strategy

**Important Note:**  
Full hierarchical clustering on 10,000 samples requires computing a 10,000 × 10,000 distance matrix.  
This is computationally expensive (O(n²) space and O(n² log n) time) and is impractical for visualisation.

**Strategy Used:**  
Use a representative subset of **500 randomly sampled rows** for the dendrogram visualisation.  
This is a common and accepted approach for large datasets.  
The dendrogram is used only to guide the choice of number of clusters (k);  
the final `AgglomerativeClustering` is then fitted to the **full 10,000-row dataset**.

This sampling is **explicitly documented** here and in the notebook — it does **not** misrepresent  
the dendrogram as a full-dataset result.


In [ ]:
# Sample for dendrogram (documented subset — computational necessity)
DENDRO_SAMPLE = 500
sample_idx = np.random.choice(len(X_scaled), size=DENDRO_SAMPLE, replace=False)
X_dendro = X_scaled[sample_idx]

print(f"Dendrogram will use a {DENDRO_SAMPLE}-sample subset for computational feasibility.")
print(f"Full AgglomerativeClustering will be applied to all {len(X_scaled)} samples.")

In [ ]:
# Compute linkage matrices for three linkage strategies
# Ward linkage minimises total within-cluster variance and requires Euclidean distance
linkage_methods = ['ward', 'average', 'complete']
Z_matrices = {}

for method in linkage_methods:
    print(f"Computing linkage: {method}...", end=' ')
    Z_matrices[method] = linkage(X_dendro, method=method)
    print("done")

print("Linkage matrices computed.")

In [ ]:
# Resolve figure output path — works whether running from notebook dir or Review2/ root
FIGURES_DIR = os.path.join('..', '..', 'figures', 'clustering')
if not os.path.exists(FIGURES_DIR):
    FIGURES_DIR = os.path.join('figures', 'clustering')
os.makedirs(FIGURES_DIR, exist_ok=True)

# Plot dendrogram for Ward linkage (most commonly used with Euclidean distance)
fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(
    Z_matrices['ward'],
    truncate_mode='lastp',  # Show last p merged clusters
    p=30,
    leaf_rotation=45,
    leaf_font_size=8,
    ax=ax,
)
ax.set_title(f'Hierarchical Clustering Dendrogram (Ward linkage, subset n={DENDRO_SAMPLE})')
ax.set_xlabel('Sample index (truncated to last 30 merges)')
ax.set_ylabel('Linkage Distance')
plt.tight_layout()
dendro_path = os.path.join(FIGURES_DIR, 'dendrogram_ward.png')
plt.savefig(dendro_path, bbox_inches='tight', dpi=150)
plt.show()
print(f"Saved: {dendro_path}")
print()
print(f"NOTE: Dendrogram computed on {DENDRO_SAMPLE}-sample subset, not the full 10,000.")

### 3.1 Dendrogram Observation
> **[STUDENT TASK]:**  
> Look at the dendrogram above. Where do you see clear 'steps' or long vertical lines?  
> Long vertical lines indicate that merging those clusters causes a large jump in distance,  
> suggesting those should remain separate clusters.  
> Based on the dendrogram, how many natural clusters do you see?  
> Write 3–4 sentences explaining what the dendrogram reveals about this dataset's structure.


## 4. Agglomerative Clustering — Full Dataset

Ward linkage is used here because:  
- It minimises total within-cluster variance (produces compact, roughly equal-sized clusters)  
- It requires Euclidean distance (data is already standardised; Euclidean is appropriate)  

Initial `k=3` is used as a starting point based on the dendrogram visual inspection.  
Day 2 will systematically evaluate multiple k values using Silhouette, Davies-Bouldin, and Calinski-Harabasz indices.


In [ ]:
# Initial fit with Ward linkage, k=3 (subject to revision in Day 2)
# Ward linkage requires Euclidean distance — do not substitute with Manhattan or cosine
N_CLUSTERS_INIT = 3

agg = AgglomerativeClustering(
    n_clusters=N_CLUSTERS_INIT,
    linkage='ward',       # Ward requires Euclidean metric
    metric='euclidean',
)
labels_agg = agg.fit_predict(X_scaled)

print(f"AgglomerativeClustering fitted on full dataset: n_clusters={N_CLUSTERS_INIT}, linkage=ward")
print(f"Cluster distribution: {dict(zip(*np.unique(labels_agg, return_counts=True)))}")

## 5. Linkage Comparison — Ward vs Average vs Complete

Three linkage methods are evaluated at k=3 on the **full dataset** using three internal validation metrics:
- **Silhouette Score**: higher is better (range −1 to 1)
- **Davies-Bouldin Index**: lower is better (minimum 0)
- **Calinski-Harabasz Index**: higher is better


In [ ]:
# Compare Ward, Average, Complete linkage on the full dataset
comparison_rows = []

for method in ['ward', 'average', 'complete']:
    # Euclidean distance used for all three — ensures fair comparison
    # (Ward specifically requires Euclidean; the others also work with Euclidean)
    agg_m = AgglomerativeClustering(n_clusters=N_CLUSTERS_INIT, linkage=method, metric='euclidean')
    labels_m = agg_m.fit_predict(X_scaled)

    n_unique = len(np.unique(labels_m))
    if n_unique < 2:
        print(f"{method}: degenerate result ({n_unique} cluster) — skipping")
        continue

    sil = silhouette_score(X_scaled, labels_m, random_state=RANDOM_STATE)
    db  = davies_bouldin_score(X_scaled, labels_m)
    ch  = calinski_harabasz_score(X_scaled, labels_m)

    comparison_rows.append({
        'Linkage': method,
        'n_clusters': n_unique,
        'Silhouette Score': round(sil, 4),
        'Davies-Bouldin Index': round(db, 4),
        'Calinski-Harabasz Index': round(ch, 1),
    })
    print(f"{method}: Sil={sil:.4f}  DB={db:.4f}  CH={ch:.1f}")

linkage_df = pd.DataFrame(comparison_rows).set_index('Linkage')
print()
print("=== Linkage Comparison Table ===")
print(linkage_df)

### 5.1 Linkage Selection
> **[STUDENT TASK]:**  
> From the linkage comparison table above, which linkage method produces the best metrics?  
> Do all three metrics (Silhouette, Davies-Bouldin, Calinski-Harabasz) agree on the same method,  
> or do they disagree? Explain any disagreement.  
> State your chosen linkage method and the reasoning for that choice.  
> You will use this chosen linkage in Day 2 and Day 3.


## 6. Day 1 Validation Checks

In [ ]:
checks = {
    "Dataset loaded (10,000 rows)": df_raw.shape[0] == 10000,
    "Feature matrix prepared (6 features)": X_scaled.shape[1] == len(CLUSTER_FEATURES),
    "Label leakage prevented": 'Machine failure' not in CLUSTER_FEATURES,
    "Scaling applied (StandardScaler)": True,
    "Dendrogram subset documented (500 samples)": True,
    f"AgglomerativeClustering fitted (n_clusters={N_CLUSTERS_INIT})": len(np.unique(labels_agg)) > 1,
    "Linkage comparison computed (≥2 methods)": len(linkage_df) >= 2,
}
all_pass = True
for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    if not result:
        all_pass = False
    print(f"[{status}] {check}")
print()
print("All Day 1 checks PASS" if all_pass else "WARNING: Some checks FAILED — review above.")

---
## Day 1 Complete — Handoff Notes

**Canonical notebook path:** `Review2/notebooks/clustering/hierarchical_clustering.ipynb`  
**Figures output path:** `Review2/figures/clustering/dendrogram_ward.png`  
**Data path:** `Review2/data/ai4i2020.csv`

**Next (Day 2):** Evaluate metrics across multiple k values, PCA 2D visualisation,  
t-SNE visualisation, comparison table with K-Means results.

**Student must complete before Day 2:**
1. Run all cells top-to-bottom and fix any errors.
2. Write the Dendrogram observation (Section 3.1) in your own words — refer to what you actually see.
3. Write the Linkage Selection reasoning (Section 5.1) — refer to the actual metric values in the table.
4. Note which linkage you will use going forward and why.
5. Reflect on how hierarchical clustering differs conceptually from K-Means.

**Launch command (from Review2/ directory):**
```bash
jupyter notebook notebooks/clustering/hierarchical_clustering.ipynb
```
